In [1]:
# =====================================================
# Cell 1: System Info & Resource Versions
# =====================================================
# This cell shows basic system info—OS version, GPU, default Python,
# pip, conda (if installed) etc. It's useful for debugging or for
# reproducibility info in a shared notebook.

!set -euxo pipefail

print("=== OS Release Info ===")
!lsb_release -a

print("\n=== GPU Info (nvidia-smi) ===")
!nvidia-smi || echo "[WARNING] nvidia-smi not found"

print("\n=== Default nvcc Version (if available) ===")
!nvcc --version || echo "[WARNING] nvcc not found"

print("\n=== Default Python & Pip Version ===")
!which python
!python --version
!pip --version

print("\n=== Conda Version (if any) ===")
!conda --version || echo "[INFO] 'conda' not yet installed."


/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
=== OS Release Info ===
/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 22.04.5 LTS
Release:	22.04
Codename:	jammy

=== GPU Info (nvidia-smi) ===
/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Mon Aug 31 13:39:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage

In [2]:
# =====================================================
# Cell 2: Install CUDA 11.8
# =====================================================
# Installs CUDA 11.8 from the official NVIDIA repo.
# We pin this version to ensure compatibility with certain libraries.

# Set environment variable for non-interactive installation
import os
os.environ['DEBIAN_FRONTEND'] = 'noninteractive'

print("---- Installing CUDA 11.8 ----")

# STEP 0) Update and install required packages
!sudo apt-get update -y
!sudo apt-get install -y gnupg curl

# STEP 1) Preconfigure keyboard settings to avoid prompts
# You can set it to a default layout, e.g., English (US)
!echo "keyboard-configuration keyboard-configuration/layout select English (US)" | sudo debconf-set-selections
!echo "keyboard-configuration keyboard-configuration/modelcode string pc105" | sudo debconf-set-selections
!echo "keyboard-configuration keyboard-configuration/layoutcode string us" | sudo debconf-set-selections

# STEP 2) Download the cuda-keyring .deb from NVIDIA's repo
!wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb

# STEP 3) Install the cuda-keyring .deb
!sudo dpkg -i cuda-keyring_1.1-1_all.deb

# STEP 4) Update apt now that the repo & key are configured
!sudo apt-get update -y

# STEP 5) Install CUDA 11.8 without interactive prompts
!sudo apt-get install -y cuda-11-8

# STEP 6) Set environment variables for CUDA
import sys
import subprocess

# Append CUDA paths to environment variables
cuda_path = "/usr/local/cuda-11.8/bin"
cuda_lib = "/usr/local/cuda-11.8/lib64"

# Update PATH and LD_LIBRARY_PATH
os.environ['PATH'] += f":{cuda_path}"
os.environ['LD_LIBRARY_PATH'] = f"{cuda_lib}:" + os.environ.get('LD_LIBRARY_PATH', '')

# Verify CUDA installation
!nvcc --version


---- Installing CUDA 11.8 ----
/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository

In [3]:
# =====================================================
# Cell 3: Double-Check CUDA 11.8 (Optional)
# =====================================================
# In Colab, environment variables may not persist across cells.
# This cell verifies that 'nvcc' is still available and at 11.8.

!export PATH="/usr/local/cuda-11.8/bin:$PATH" && \
nvcc --version


/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2022 NVIDIA Corporation
Built on Wed_Sep_21_10:33:58_PDT_2022
Cuda compilation tools, release 11.8, V11.8.89
Build cuda_11.8.r11.8/compiler.31833905_0


In [4]:
# =====================================================
# Cell 4: Install Miniconda using condacolab
# =====================================================
# We use `condacolab` to install and configure a Miniconda environment
# inside Colab. Once installed, we still need to "source" and activate
# it in bash cells to use conda commands.

!pip install -q condacolab
import condacolab
condacolab.install_miniconda()


/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)

📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

✨🍰✨ Everything looks OK!


Miniconda is subject to terms of service: https://anaconda.com/legal/terms/terms-of-service


In [8]:
%%bash
set -ex

# Make conda commands available in this shell
source /usr/local/etc/profile.d/conda.sh

# ยอมรับ Terms of Service ของ Anaconda เพื่อปลดล็อก Error
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Prepend /usr/local/cuda-11.8 to PATH
export PATH="/usr/local/cuda-11.8/bin:$PATH"
export LD_LIBRARY_PATH="/usr/local/cuda-11.8/lib64:$LD_LIBRARY_PATH"

# Create & activate the conda environment
conda create -y -n trellis python=3.11
conda activate trellis

# Install PyTorch (2.3.0) + CUDA 11.8 build
conda install -y pytorch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 \
               pytorch-cuda=11.8 -c pytorch -c nvidia

# Install flash-attn 2.5.8 build for PyTorch 2.3 + CUDA 11.8
pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.5.8/flash_attn-2.5.8+cu118torch2.3cxx11abiFALSE-cp311-cp311-linux_x86_64.whl

# Install spconv for CUDA 11.8
pip install spconv-cu118

# Install kaolin
pip install kaolin==0.17.0 -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.3.0_cu118.html

# Pillow < 11.0 if needed for certain dependencies
pip install --force-reinstall "pillow<11.0"

# Install Gradio + custom gradio_litmodel3d
pip install gradio==4.44.1 gradio_litmodel3d==0.0.1

echo "[INFO] 'trellis' environment created and packages installed."

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ | / - \ | / - \ | / - \ | / - \ | / - \ | done
Channels:
 - defaults
Platform: linux-64
Solving environment: - \ done

## Package Plan ##

  environment location: /usr/local/envs/trellis

  added / updated specs:
    - python=3.11


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.8.13  |       h06a4308_0         107 KB
    libnsl-2.0.0               |       h5eee18b_0          31 KB
    ncurses-6.6                |       hfaaeb4e_0         893 KB
    openssl-3.5.8              |       h1b28b03_0         5.5 MB
    packaging-26.3             |  py311h06a4308_0         380 KB
    pip-26.2.1                 |     py

bash: /usr/local/lib/libtinfo.so.6: no version information available (required by bash)
+ source /usr/local/etc/profile.d/conda.sh
++ export CONDA_EXE=/usr/local/bin/conda
++ CONDA_EXE=/usr/local/bin/conda
++ export _CONDA_EXE=/usr/local/bin/conda
++ _CONDA_EXE=/usr/local/bin/conda
++ export _CE_M=
++ _CE_M=
++ export _CE_CONDA=
++ _CE_CONDA=
++ export CONDA_PYTHON_EXE=/usr/local/bin/python
++ CONDA_PYTHON_EXE=/usr/local/bin/python
++ export _CONDA_ROOT=/usr/local
++ _CONDA_ROOT=/usr/local
++ '[' -z '' ']'
++ export CONDA_SHLVL=0
++ CONDA_SHLVL=0
++ '[' -n '' ']'
++++ dirname /usr/local/bin/conda
+++ dirname /usr/local/bin
++ PATH=/usr/local/condabin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin:/usr/local/cuda-11.8/bin
++ export PATH
++ '[' -z '' ']'
++ PS1=
+ conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
+ local cmd=tos
+ case "$cmd" in
+ __conda_exe tos accept

In [9]:
# =====================================================
# Cell 6: System Dev Tools & Python Build Libraries
# =====================================================
# Installs system-level development tools: build-essential, cmake, ninja, nvidia-cuda-toolkit
# Then re-activates the 'trellis' environment and upgrades pip, setuptools, wheel, ninja, Cython.

%%bash
set -ex

apt-get update -y
apt-get install -y build-essential cmake ninja-build nvidia-cuda-toolkit

# Make conda available + activate 'trellis'
source /usr/local/etc/profile.d/conda.sh
conda activate trellis

# Upgrade build libraries in Python
pip install --upgrade pip setuptools wheel ninja Cython

echo "[INFO] System dev tools + Python build libs installed."


Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,263 kB]
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,619 kB]
Get:14 http://archive.ubuntu.com/ubuntu

bash: /usr/local/lib/libtinfo.so.6: no version information available (required by bash)
+ apt-get update -y
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
+ apt-get install -y build-essential cmake ninja-build nvidia-cuda-toolkit
+ source /usr/local/etc/profile.d/conda.sh
++ export CONDA_EXE=/usr/local/bin/conda
++ CONDA_EXE=/usr/local/bin/conda
++ export _CONDA_EXE=/usr/local/bin/conda
++ _CONDA_EXE=/usr/local/bin/conda
++ export _CE_M=
++ _CE_M=
++ export _CE_CONDA=
++ _CE_CONDA=
++ export CONDA_PYTHON_EXE=/usr/local/bin/python
++ CONDA_PYTHON_EXE=/usr/local/bin/python
++ export _CONDA_ROOT=/usr/local
++ _CONDA_ROOT=/usr/local
++ '[' -z '' ']'
++ export CONDA_SHLVL=0
++ CONDA_SHLVL=0
++ '[' -n '' ']'
++++ dirname /usr/local/bin/conda
+++ dirname /usr/local/bin
++ PATH=/usr/local/condabin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/b

May crash at this point. if it does, just resume in the cell below


In [10]:
# =====================================================
# Cell 7: Clone & Setup Microsoft/TRELLIS
# =====================================================
# 1) Source conda script
# 2) Prepend CUDA 11.8
# 3) Activate trellis
# 4) Clean leftover submodule directories
# 5) Clone the MS/TRELLIS repo
# 6) Create 'assets/example_image' folder so app.py won't crash
# 7) Run setup.sh with all relevant flags
# 8) Patch app.py to launch share=True (so you can share the Gradio link)
# 9) Confirm which nvcc is being used

%%bash
set -ex

# Source conda script
source /usr/local/etc/profile.d/conda.sh

# Prepend /usr/local/cuda-11.8 to PATH
export PATH="/usr/local/cuda-11.8/bin:$PATH"
export LD_LIBRARY_PATH="/usr/local/cuda-11.8/lib64:$LD_LIBRARY_PATH"

# Activate 'trellis'
conda activate trellis

# Remove leftover submodule directories
rm -rf /tmp/extensions
rm -rf /tmp/pip-req-build-*

# Clone official MS/TRELLIS repo
rm -rf /content/TRELLIS
git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git /content/TRELLIS
cd /content/TRELLIS
git submodule update --init --recursive

# Create missing folder so app.py won't crash
mkdir -p /content/TRELLIS/assets/example_image

# Run setup.sh
bash ./setup.sh --basic --flash-attn --spconv --mipgaussian --nvdiffrast --kaolin --demo || \
  echo "[WARNING] Some submodules likely failed to build, but that's often normal in Colab."

# Patch app.py => use demo.launch(share=True)
sed -i 's/demo.launch()$/demo.launch(share=True)/' app.py

# Confirm which nvcc is being used now
nvcc --version

echo "[INFO] TRELLIS environment set up; 'app.py' patched."


Submodule path 'trellis/representations/mesh/flexicubes': checked out '815e075a2a400d06c48d94c347674344ed6ae5c5'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.5/29.5 MB 109.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 MB 49.6 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 137.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 143.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 126.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 134.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.5/745.5 kB 36.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 35.5 MB/s  0:00:08
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 93.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.0/146.0 MB 61.5 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.

bash: /usr/local/lib/libtinfo.so.6: no version information available (required by bash)
+ source /usr/local/etc/profile.d/conda.sh
++ export CONDA_EXE=/usr/local/bin/conda
++ CONDA_EXE=/usr/local/bin/conda
++ export _CONDA_EXE=/usr/local/bin/conda
++ _CONDA_EXE=/usr/local/bin/conda
++ export _CE_M=
++ _CE_M=
++ export _CE_CONDA=
++ _CE_CONDA=
++ export CONDA_PYTHON_EXE=/usr/local/bin/python
++ CONDA_PYTHON_EXE=/usr/local/bin/python
++ export _CONDA_ROOT=/usr/local
++ _CONDA_ROOT=/usr/local
++ '[' -z '' ']'
++ export CONDA_SHLVL=0
++ CONDA_SHLVL=0
++ '[' -n '' ']'
++++ dirname /usr/local/bin/conda
+++ dirname /usr/local/bin
++ PATH=/usr/local/condabin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin:/usr/local/cuda-11.8/bin
++ export PATH
++ '[' -z '' ']'
++ PS1=
+ export PATH=/usr/local/cuda-11.8/bin:/usr/local/condabin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbi

You may see warnings in the out of cell below. safe to ignore.

In [17]:
%%bash
set -ex

cd /content
if [ ! -d "/content/mip-splatting" ]; then
  echo "[INFO] 'mip-splatting' folder not found, cloning now..."
  git clone --recurse-submodules https://github.com/autonomousvision/mip-splatting.git
  cd mip-splatting
  git submodule update --init --recursive
  cd ..
else
  echo "[INFO] 'mip-splatting' folder already exists, skipping clone."
fi

# Install system dependencies
apt-get update -y
apt-get install -y libgl1-mesa-dev build-essential cmake ninja-build

# Activate environment 'trellis'
source /usr/local/etc/profile.d/conda.sh
conda activate trellis

# 1. Remove conflicting Conda MKL packages shadowing PyTorch
conda remove -y --force mkl mkl-include intel-openmp || true

# 2. Force reinstall torch via pip to restore clean internal library bindings
pip install --force-reinstall --no-deps torch torchvision --index-url https://download.pytorch.org/whl/cu118

# 3. Export CUDA variables strictly inside the environment runtime
export CUDA_HOME="/usr/local/cuda-11.8"
export PATH="/usr/local/cuda-11.8/bin:$PATH"
export LD_LIBRARY_PATH="/usr/local/envs/trellis/lib/python3.11/site-packages/torch/lib:/usr/local/cuda-11.8/lib64:$LD_LIBRARY_PATH"

# 4. Build and install diff_gaussian_rasterization
cd /content/mip-splatting/submodules/diff-gaussian-rasterization
pip install . --no-build-isolation -v

echo "[INFO] diff_gaussian_rasterization installed successfully."

[INFO] 'mip-splatting' folder already exists, skipping clone.
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
build-essential is already the newest version (12.9ubuntu3).
nin

bash: /usr/local/lib/libtinfo.so.6: no version information available (required by bash)
+ cd /content
+ '[' '!' -d /content/mip-splatting ']'
+ echo '[INFO] '\''mip-splatting'\'' folder already exists, skipping clone.'
+ apt-get update -y
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
+ apt-get install -y libgl1-mesa-dev build-essential cmake ninja-build
+ source /usr/local/etc/profile.d/conda.sh
++ export CONDA_EXE=/usr/local/bin/conda
++ CONDA_EXE=/usr/local/bin/conda
++ export _CONDA_EXE=/usr/local/bin/conda
++ _CONDA_EXE=/usr/local/bin/conda
++ export _CE_M=
++ _CE_M=
++ export _CE_CONDA=
++ _CE_CONDA=
++ export CONDA_PYTHON_EXE=/usr/local/bin/python
++ CONDA_PYTHON_EXE=/usr/local/bin/python
++ export _CONDA_ROOT=/usr/local
++ _CONDA_ROOT=/usr/local
++ '[' -z '' ']'
++ export CONDA_SHLVL=0
++ CONDA_SHLVL=0
++ '[' -n '' ']'
++++ dirname /usr/l

**RUN TRELLIS IN NGROK (PROVIDE YOUR NGROK API KEY)**\
if you run the code below, dont forget to edit and put in your nGrok Auth token\
Also, by default, we load kaiser-app.py \
if you want original app.py, uncomment that line and comment out kaiser-app.py


In [ ]:
# =====================================================
# Cell 9: Launch TRELLIS App with Ngrok (Gradio 4.x Patched)
# =====================================================
import os, re, glob, time, subprocess

TARGET_DIR = "/content/TRELLIS"
os.makedirs(TARGET_DIR, exist_ok=True)
kaiser_app_path = os.path.join(TARGET_DIR, "kaiser-app.py")

# สคริปต์ kaiser-app.py เวอร์ชันแก้ไข Syntax Gradio 4.x + ลำดับ Arguments
FIXED_APP_CODE = r'''import gradio_client.utils
_orig_json_schema_to_python_type = gradio_client.utils._json_schema_to_python_type

def _patched_json_schema_to_python_type(schema, defs=None):
    if not isinstance(schema, dict):
        return "Any"
    return _orig_json_schema_to_python_type(schema, defs)

gradio_client.utils._json_schema_to_python_type = _patched_json_schema_to_python_type

import gradio as gr
import gradio.themes as themes
from gradio_litmodel3d import LitModel3D

import os
import shutil
from typing import List, Tuple, Literal
import torch
import numpy as np
import imageio
from easydict import EasyDict as edict
from PIL import Image

from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.representations import Gaussian, MeshExtractResult
from trellis.utils import render_utils, postprocessing_utils

MAX_SEED = np.iinfo(np.int32).max
TMP_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "tmp")
os.makedirs(TMP_DIR, exist_ok=True)

pipeline = TrellisImageTo3DPipeline.from_pretrained("JeffreyXiang/TRELLIS-image-large")
pipeline.cuda()

def start_session(req: gr.Request):
    user_dir = os.path.join(TMP_DIR, str(req.session_hash))
    os.makedirs(user_dir, exist_ok=True)

def end_session(req: gr.Request):
    user_dir = os.path.join(TMP_DIR, str(req.session_hash))
    if os.path.exists(user_dir):
        shutil.rmtree(user_dir)

def preprocess_image(image: Image.Image) -> Image.Image:
    if image is None:
        return None
    return pipeline.preprocess_image(image)

def preprocess_images(images: List[Tuple[Image.Image, str]]) -> List[Image.Image]:
    if not images:
        return []
    img_list = [img_tuple[0] for img_tuple in images]
    return [pipeline.preprocess_image(img) for img in img_list]

def pack_state(gs: Gaussian, mesh: MeshExtractResult) -> dict:
    return {
        'gaussian': {
            **gs.init_params,
            '_xyz': gs._xyz.cpu().numpy(),
            '_features_dc': gs._features_dc.cpu().numpy(),
            '_scaling': gs._scaling.cpu().numpy(),
            '_rotation': gs._rotation.cpu().numpy(),
            '_opacity': gs._opacity.cpu().numpy(),
        },
        'mesh': {
            'vertices': mesh.vertices.cpu().numpy(),
            'faces': mesh.faces.cpu().numpy(),
        },
    }

def unpack_state(state: dict) -> Tuple[Gaussian, edict]:
    gs = Gaussian(
        aabb=state['gaussian']['aabb'],
        sh_degree=state['gaussian']['sh_degree'],
        mininum_kernel_size=state['gaussian']['mininum_kernel_size'],
        scaling_bias=state['gaussian']['scaling_bias'],
        opacity_bias=state['gaussian']['opacity_bias'],
        scaling_activation=state['gaussian']['scaling_activation'],
    )
    gs._xyz = torch.tensor(state['gaussian']['_xyz'], device='cuda')
    gs._features_dc = torch.tensor(state['gaussian']['_features_dc'], device='cuda')
    gs._scaling = torch.tensor(state['gaussian']['_scaling'], device='cuda')
    gs._rotation = torch.tensor(state['gaussian']['_rotation'], device='cuda')
    gs._opacity = torch.tensor(state['gaussian']['_opacity'], device='cuda')

    mesh = edict(
        vertices=torch.tensor(state['mesh']['vertices'], device='cuda'),
        faces=torch.tensor(state['mesh']['faces'], device='cuda'),
    )
    return gs, mesh

def get_seed(randomize_seed: bool, seed: int) -> int:
    return np.random.randint(0, MAX_SEED) if randomize_seed else seed

def image_to_3d(
    image: Image.Image,
    multiimages: List[Tuple[Image.Image, str]],
    is_multiimage: bool,
    seed: int,
    ss_guidance_strength: float,
    ss_sampling_steps: int,
    slat_guidance_strength: float,
    slat_sampling_steps: int,
    multiimage_algo: Literal["multidiffusion", "stochastic"],
    req: gr.Request = None,
    *args, **kwargs
) -> Tuple[dict, str]:
    session_hash = req.session_hash if req is not None else "default_session"
    user_dir = os.path.join(TMP_DIR, str(session_hash))
    os.makedirs(user_dir, exist_ok=True)

    if not is_multiimage:
        outputs = pipeline.run(
            image,
            seed=seed,
            formats=["gaussian", "mesh"],
            preprocess_image=False,
            sparse_structure_sampler_params={
                "steps": ss_sampling_steps,
                "cfg_strength": ss_guidance_strength,
            },
            slat_sampler_params={
                "steps": slat_sampling_steps,
                "cfg_strength": slat_guidance_strength,
            },
        )
    else:
        outputs = pipeline.run_multi_image(
            [img_tuple[0] for img_tuple in multiimages],
            seed=seed,
            formats=["gaussian", "mesh"],
            preprocess_image=False,
            sparse_structure_sampler_params={
                "steps": ss_sampling_steps,
                "cfg_strength": ss_guidance_strength,
            },
            slat_sampler_params={
                "steps": slat_sampling_steps,
                "cfg_strength": slat_guidance_strength,
            },
            mode=multiimage_algo,
        )

    color_frames = render_utils.render_video(outputs['gaussian'][0], num_frames=120)['color']
    geo_frames = render_utils.render_video(outputs['mesh'][0], num_frames=120)['normal']

    video_frames = [np.concatenate([c, g], axis=1) for c, g in zip(color_frames, geo_frames)]
    video_path = os.path.join(user_dir, 'sample.mp4')
    imageio.mimsave(video_path, video_frames, fps=15)

    state = pack_state(outputs['gaussian'][0], outputs['mesh'][0])
    torch.cuda.empty_cache()
    return state, video_path

def extract_glb(state: dict, mesh_simplify: float, texture_size: int, req: gr.Request = None, *args, **kwargs) -> Tuple[str, str]:
    session_hash = req.session_hash if req is not None else "default_session"
    user_dir = os.path.join(TMP_DIR, str(session_hash))
    gs, mesh = unpack_state(state)

    glb = postprocessing_utils.to_glb(
        gs,
        mesh,
        simplify=mesh_simplify,
        texture_size=texture_size,
        verbose=False
    )
    glb_path = os.path.join(user_dir, 'sample.glb')
    glb.export(glb_path)

    torch.cuda.empty_cache()
    return glb_path, glb_path

def extract_gaussian(state: dict, req: gr.Request = None, *args, **kwargs) -> Tuple[str, str]:
    session_hash = req.session_hash if req is not None else "default_session"
    user_dir = os.path.join(TMP_DIR, str(session_hash))
    gs, _ = unpack_state(state)

    ply_path = os.path.join(user_dir, 'sample.ply')
    gs.save_ply(ply_path)

    torch.cuda.empty_cache()
    return ply_path, ply_path

def prepare_multi_example() -> List[Image.Image]:
    if not os.path.exists("assets/example_multi_image"):
        return []

    cases = set(fn.split('_')[0] for fn in os.listdir("assets/example_multi_image"))
    examples = []
    for case in cases:
        rows = []
        for i in range(1, 4):
            fn = f"assets/example_multi_image/{case}_{i}.png"
            if not os.path.exists(fn):
                continue
            img = Image.open(fn).convert("RGBA")
            W, H = img.size
            new_w = int(W / H * 512)
            rows.append(np.array(img.resize((new_w, 512))))
        if rows:
            combined = np.concatenate(rows, axis=1)
            examples.append(Image.fromarray(combined))
    return examples

def split_image(image: Image.Image) -> List[Image.Image]:
    arr = np.array(image)
    alpha = arr[..., 3]
    alpha_mask = np.any(alpha > 0, axis=0)

    start_positions = np.where(~alpha_mask[:-1] & alpha_mask[1:])[0].tolist()
    end_positions = np.where(alpha_mask[:-1] & ~alpha_mask[1:])[0].tolist()

    subimages = []
    for s, e in zip(start_positions, end_positions):
        segment = Image.fromarray(arr[:, s:e+1])
        subimages.append(preprocess_image(segment))

    return subimages

def build_interface():
    with gr.Blocks(
        title="TRELLIS: Image to 3D",
        theme=themes.Soft(),
        css=None,
        analytics_enabled=False
    ) as demo:

        gr.Markdown("""
        # Image to 3D with TRELLIS
        Convert one or multiple images into a 3D mesh + Gaussian representation (via Microsoft TRELLIS).
        """)

        with gr.Row():
            with gr.Column():
                with gr.Tabs() as input_tabs:
                    with gr.Tab(label="Single Image", id=0) as single_tab:
                        image_prompt = gr.Image(
                            label="Image Prompt (RGBA)",
                            format="png",
                            image_mode="RGBA",
                            type="pil",
                            height=300
                        )

                    with gr.Tab(label="Multiple Images", id=1) as multi_tab:
                        multiimage_prompt = gr.Gallery(
                            label="Multi-Image Prompt",
                            format="png",
                            type="pil",
                            height=300,
                            columns=3
                        )

                with gr.Accordion("Generation Settings", open=False):
                    seed = gr.Slider(0, MAX_SEED, value=0, step=1, label="Seed")
                    randomize_seed = gr.Checkbox(value=True, label="Randomize Seed")

                    gr.Markdown("**Stage 1: Sparse Structure**")
                    with gr.Row():
                        ss_guidance_strength = gr.Slider(0.0, 10.0, value=7.5, step=0.1, label="Guidance Strength")
                        ss_sampling_steps = gr.Slider(1, 50, value=12, step=1, label="Sampling Steps")

                    gr.Markdown("**Stage 2: Structured Latent**")
                    with gr.Row():
                        slat_guidance_strength = gr.Slider(0.0, 10.0, value=3.0, step=0.1, label="Guidance Strength")
                        slat_sampling_steps = gr.Slider(1, 50, value=12, step=1, label="Sampling Steps")

                    multiimage_algo = gr.Radio(
                        choices=["stochastic", "multidiffusion"],
                        value="stochastic",
                        label="Multi-Image Mode"
                    )

                generate_btn = gr.Button("Generate 3D Model", variant="primary")

                with gr.Accordion("Export Settings (GLB)", open=False):
                    mesh_simplify = gr.Slider(0.9, 0.98, value=0.95, step=0.01, label="Mesh Simplification")
                    texture_size = gr.Slider(512, 2048, value=1024, step=512, label="Texture Resolution")

                with gr.Row():
                    extract_glb_btn = gr.Button("Export as GLB", interactive=False)
                    extract_gs_btn = gr.Button("Export as Gaussian", interactive=False)

            with gr.Column():
                video_output = gr.Video(label="Preview (Color + Geometry)", autoplay=True, loop=True, height=300)
                model_output = LitModel3D(label="Extracted 3D Model", exposure=10.0, height=300)

                with gr.Row():
                    download_glb = gr.DownloadButton(label="Download .GLB", interactive=False)
                    download_gs = gr.DownloadButton(label="Download .PLY", interactive=False)

        is_multiimage = gr.State(False)
        output_buf = gr.State()

        with gr.Row(visible=True) as single_example_row:
            examples_list = [
                os.path.join("assets/example_image", fn)
                for fn in os.listdir("assets/example_image")
            ] if os.path.exists("assets/example_image") else []

            examples = gr.Examples(
                examples=examples_list,
                inputs=[image_prompt],
                fn=preprocess_image,
                outputs=[image_prompt],
                run_on_click=True,
                examples_per_page=64,
                label="Single Image Examples"
            )

        with gr.Row(visible=False) as multi_example_row:
            multi_examples_list = prepare_multi_example()
            examples_multi = gr.Examples(
                examples=multi_examples_list,
                inputs=[image_prompt],
                fn=split_image,
                outputs=[multiimage_prompt],
                run_on_click=True,
                examples_per_page=8,
                label="Multi-View Examples"
            )

        demo.load(start_session)
        demo.unload(end_session)

        def single_tab_fn():
            return False, gr.update(visible=True), gr.update(visible=False)

        def multi_tab_fn():
            return True, gr.update(visible=False), gr.update(visible=True)

        single_tab.select(single_tab_fn, outputs=[is_multiimage, single_example_row, multi_example_row])
        multi_tab.select(multi_tab_fn, outputs=[is_multiimage, single_example_row, multi_example_row])

        image_prompt.upload(preprocess_image, inputs=[image_prompt], outputs=[image_prompt])
        multiimage_prompt.upload(preprocess_images, inputs=[multiimage_prompt], outputs=[multiimage_prompt])

        generate_btn.click(
            get_seed,
            inputs=[randomize_seed, seed],
            outputs=[seed]
        ).then(
            image_to_3d,
            inputs=[
                image_prompt,
                multiimage_prompt,
                is_multiimage,
                seed,
                ss_guidance_strength,
                ss_sampling_steps,
                slat_guidance_strength,
                slat_sampling_steps,
                multiimage_algo
            ],
            outputs=[output_buf, video_output]
        ).then(
            lambda: (gr.update(interactive=True), gr.update(interactive=True)),
            outputs=[extract_glb_btn, extract_gs_btn]
        )

        video_output.clear(
            lambda: (gr.update(interactive=False), gr.update(interactive=False)),
            outputs=[extract_glb_btn, extract_gs_btn],
        )

        extract_glb_btn.click(
            extract_glb,
            inputs=[output_buf, mesh_simplify, texture_size],
            outputs=[model_output, download_glb]
        ).then(
            lambda: gr.update(interactive=True),
            outputs=[download_glb]
        )

        extract_gs_btn.click(
            extract_gaussian,
            inputs=[output_buf],
            outputs=[model_output, download_gs]
        ).then(
            lambda: gr.update(interactive=True),
            outputs=[download_gs]
        )

        model_output.clear(
            lambda: gr.update(interactive=False),
            outputs=[download_glb],
        )

        return demo

if __name__ == "__main__":
    demo = build_interface()
    demo.launch(show_api=False)
'''

with open(kaiser_app_path, "w", encoding="utf-8") as f:
    f.write(FIXED_APP_CODE)

APP_FILE = "kaiser-app.py"
NGROK_AUTH = "2nmFL8KW47qZ2cLOKfpYSHvvbmT_jEBbtqqq1hHCZWHW1ifk"

# Step 1: Install pyngrok
subprocess.run(["pip", "install", "-q", "pyngrok"], check=True)

# Step 2: Restore PyTorch 2.4.0 & Kaolin ABI core packages
print("[2/6] Restoring PyTorch 2.4.0 & Kaolin ABI in 'trellis' environment...")
subprocess.run([
    "conda", "run", "-n", "trellis",
    "pip", "install", "torch==2.4.0", "torchvision==0.19.0", "--index-url", "https://download.pytorch.org/whl/cu121"
], check=True)

subprocess.run([
    "conda", "run", "-n", "trellis",
    "pip", "install", "kaolin==0.16.0", "-f", "https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.4.0_cu121.html"
], check=True)

# Step 3: Install nvdiffrast
print("[3/6] Installing nvdiffrast from NVlabs GitHub...")
subprocess.run([
    "conda", "run", "-n", "trellis",
    "pip", "install", "--no-build-isolation", "git+https://github.com/NVlabs/nvdiffrast.git"
], check=True)

# Step 4: Install UI & NLP dependencies
print("[4/6] Installing UI & NLP dependencies...")
subprocess.run([
    "conda", "run", "-n", "trellis",
    "pip", "install", "huggingface_hub==0.25.2", "transformers==4.46.3", "tokenizers==0.20.3", "gradio==4.44.1", "gradio_litmodel3d==0.0.1"
], check=True)

# Step 5: Patch backends and Gradio client bugs
print("[5/6] Patching backends and Gradio client bugs...")
site_packages = "/usr/local/envs/trellis/lib/python3.11/site-packages"

flash_dir = os.path.join(site_packages, "flash_attn")
os.makedirs(flash_dir, exist_ok=True)
with open(os.path.join(flash_dir, "__init__.py"), "w") as f:
    f.write('__version__ = "2.6.3"\n')

gc_utils = os.path.join(site_packages, "gradio_client/utils.py")
if os.path.exists(gc_utils):
    with open(gc_utils, "r", encoding="utf-8") as f:
        content = f.read()
    if "def _json_schema_to_python_type(" in content and "if not isinstance(schema, dict):" not in content:
        content = content.replace(
            "def _json_schema_to_python_type(schema",
            "def _json_schema_to_python_type(schema:\n    if not isinstance(schema, dict):\n        return 'Any'\n    schema"
        ).replace("schema:\n    if", "schema,\n    if")
        with open(gc_utils, "w", encoding="utf-8") as f:
            f.write(content)

# Step 6: Setup Ngrok Tunnel & Launch
print("[6/6] Setting up Ngrok Tunnel...")
from pyngrok import ngrok

ngrok.kill()
ngrok.set_auth_token(NGROK_AUTH)

tunnel = ngrok.connect("127.0.0.1:7860")
public_url = tunnel.public_url

print(f"\n[INFO] Ngrok Tunnel active: {public_url}")
print("[INFO] Loading TRELLIS models into GPU VRAM... Please wait.\n")

env = os.environ.copy()
env["ATTN_BACKEND"] = "sdpa"
env["SPARSE_ATTN_BACKEND"] = "sdpa"
env["PYTHONUNBUFFERED"] = "1"

app_path = os.path.join(TARGET_DIR, APP_FILE)

p = subprocess.Popen(
    ["conda", "run", "-n", "trellis", "python", "-u", app_path],
    cwd=TARGET_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=env,
    text=True
)

launch_detected = False

try:
    for line in iter(p.stdout.readline, ""):
        if not line:
            break
        print(line, end="")
        if "Running on local URL:" in line or "http://127.0.0.1:7860" in line or "http://0.0.0.0:7860" in line:
            launch_detected = True
            print(f"\n=======================================================")
            print(f"[SUCCESS] {APP_FILE} is live!")
            print(f"Open Web UI here: {public_url}")
            print(f"=======================================================\n")

except KeyboardInterrupt:
    print("[INFO] Interrupted by user. Stopping app and ngrok.")
    p.kill()
    ngrok.kill()
finally:
    p.wait()

if not launch_detected:
    print(f"\n[INFO] Access via Ngrok URL:\n{public_url}\n")

[2/6] Restoring PyTorch 2.4.0 & Kaolin ABI in 'trellis' environment...
[3/6] Installing nvdiffrast from NVlabs GitHub...
[4/6] Installing UI & NLP dependencies...


t=2026-08-31T15:03:41+0000 lvl=warn msg="ngrok config file found at both XDG and legacy locations, using XDG location" xdg_path=/root/.config/ngrok/ngrok.yml legacy_path=/root/.ngrok2/ngrok.yml


[5/6] Patching backends and Gradio client bugs...
[6/6] Setting up Ngrok Tunnel...

[INFO] Ngrok Tunnel active: https://de4b-34-124-149-139.ngrok-free.app
[INFO] Loading TRELLIS models into GPU VRAM... Please wait.



**RUN WITH CLOUDFLARE - NO AUTH KEY NEEDED**

In [ ]:
# =====================================================
# Cell 10: Launch TRELLIS App with Cloudflare (show URL after launch)
# =====================================================
# 0) Download kaiser-app.py from your GitHub (if not present)
# 1) Install cloudflared (if not already)
# 2) (Re)install Gradio in 'trellis' environment
# 3) Start a Cloudflare "Quick Tunnel" on localhost:7860
# 4) Launch either 'app.py' or 'kaiser-app.py' in 'trellis' environment
# 5) Print the tunnel URL AFTER the app is actually up.

import subprocess
import time
import re
import os

# ============== STEP 0) Download kaiser-app.py ==============
TARGET_DIR = "/content/TRELLIS"
kaiser_app_path = os.path.join(TARGET_DIR, "kaiser-app.py")

print("[INFO] Downloading kaiser-app.py from GitHub into /content/TRELLIS ...")
KAISER_APP_URL = "https://raw.githubusercontent.com/jackel27/Trellis-Colab/main/kaiser-app.py"
subprocess.run(["wget", "-q", "-O", kaiser_app_path, KAISER_APP_URL], check=True)

# ============== USER CHOICE ==============
# Uncomment exactly ONE of the following lines
# to select which app file you want to run.

#APP_FILE = "app.py"
APP_FILE = "kaiser-app.py"
# =========================================

# Step 1) Install/Update cloudflared via the latest .deb for Linux (amd64)
print("[INFO] Installing/Updating cloudflared...")
try:
    # Download the latest .deb from GitHub
    subprocess.run([
        "wget", "-q", "-O", "cloudflared-linux-amd64.deb",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"
    ], check=True)
    # Attempt dpkg install
    subprocess.run(["sudo", "dpkg", "-i", "cloudflared-linux-amd64.deb"], check=True)
except subprocess.CalledProcessError:
    print("[WARNING] cloudflared may already be installed or dpkg encountered an error.")

# Step 2) (Re)install Gradio in 'trellis' environment
print("[INFO] Ensuring Gradio is installed in 'trellis' environment...")
subprocess.run([
    "conda", "run", "-n", "trellis",
    "pip", "install", "gradio==4.44.1", "gradio_litmodel3d==0.0.1"
], check=True)

# Step 3) Start cloudflared in a subprocess to create a Quick Tunnel from :7860
print("[INFO] Starting cloudflared (Quick Tunnel) on port 7860...")
cloudflare_cmd = ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"]
proc_cloudflare = subprocess.Popen(
    cloudflare_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Give cloudflared a few seconds to initialize logs
time.sleep(3)

# We'll store the discovered tunnel URL in `tunnel_url`
tunnel_url = None

def extract_cloudflare_url(log_line: str):
    # We look for "https://xxxxx.trycloudflare.com"
    match = re.search(r"https://.*?\.trycloudflare\.com\S*", log_line)
    return match.group(0) if match else None

print("[INFO] Searching cloudflared logs for the public URL...")

# Check up to 10 seconds for a log line containing the tunnel URL
start_time = time.time()
while (time.time() - start_time) < 10:
    if proc_cloudflare.poll() is not None:
        # cloudflared has exited or failed
        break

    line = proc_cloudflare.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue

    possible_url = extract_cloudflare_url(line)
    if possible_url:
        tunnel_url = possible_url
        break

if tunnel_url:
    print(f"[INFO] Found Cloudflare Tunnel URL => {tunnel_url}")
else:
    print("[WARNING] Could not parse the Cloudflare URL from logs yet.")

# Step 4) Launch either app.py or kaiser-app.py in the 'trellis' environment, unbuffered
app_path = os.path.join(TARGET_DIR, APP_FILE)
print(f"[INFO] Launching TRELLIS app ({APP_FILE}) in the 'trellis' environment...")
p_app = subprocess.Popen(
    ["conda", "run", "-n", "trellis", "python", "-u", app_path],
    cwd=TARGET_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Step 5) Stream logs from the selected app until it prints "Running on local URL:"
launch_detected = False
try:
    for line in iter(p_app.stdout.readline, ""):
        if not line:
            break

        print(line, end="")

        # If the app prints "Running on local URL:", we assume the server is up
        if "Running on local URL:" in line:
            launch_detected = True
            if tunnel_url:
                print(f"\n[INFO] {APP_FILE} is live! Access via Cloudflare:\n{tunnel_url}\n")
            else:
                print("\n[WARNING] We have not found a Cloudflare URL, but the app claims to be running.\n")

except KeyboardInterrupt:
    print("[INFO] Interrupted by user. Stopping the app.")
    p_app.kill()
finally:
    p_app.wait()

# If the app never printed "Running on local URL:", show the URL (if any)
if not launch_detected and tunnel_url:
    print("\n[INFO] The app didn't print 'Running on local URL:' but here's the Cloudflare URL anyway:")
    print(tunnel_url)

print("[INFO] Done. If you'd like to relaunch, just re-run this cell or the entire notebook.")
